# MLPerf LLM Benchmark
Runs MLPerf Inference benchmarks for text generation using the **corespan/inference** repo.
Uses the actual `language/<model>/main.py` + `SUT_VLLM.py` pipeline.
`mlperf_loadgen` is pre-built in this image.

**Run cells top to bottom. Edit only Cell 2.**

## Cell 1 — GPU Verification

In [ ]:
import torch
print('=== GPU Status ===')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i} : {p.name}  {p.total_memory/1e9:.1f} GB')
    _ = torch.rand(512,512,device='cuda') @ torch.rand(512,512,device='cuda')
    torch.cuda.synchronize()
    print('Smoke test : PASSED')
else:
    print('ERROR: LLM benchmark requires a GPU. Stopping.')


## Cell 2 — Configuration

In [ ]:
# ── Edit these ──────────────────────────────────────────────────────────────
MODEL_NAME  = 'TinyLlama-1.1B-Chat-v1.0'   # Quick default
# Other models on NFS:
#   'Llama-3.1-8B'                 -> uses language/llama3.1-8b
#   'Qwen2.5-7B'                   -> uses language/llama3.1-8b (compatible)
#   'Qwen2.5-7B-Instruct'          -> uses language/llama3.1-8b
#   'DeepSeek-R1-Distill-Qwen-1.5B'  -> uses language/llama3.1-8b
#   'DeepSeek-R1-Distill-Qwen-32B' -> uses language/llama3.1-8b
#   'llama3.1-70B'                 -> uses language/llama2-70b

DATASET_FILE   = 'openorca-train-10.jsonl'  # openorca-train-10 / 100 / full
SCENARIO       = 'Offline'
TOTAL_SAMPLES  = 10          # Samples to run (max = lines in dataset file)
DTYPE          = 'float16'
TENSOR_PARALLEL = 1          # Match number of GPUs on this node

# ── NFS paths ────────────────────────────────────────────────────────────────
NFS_MODEL_PATH  = f'/data/LLMInference/models/{MODEL_NAME}'
NFS_DATASET_DIR = '/data/LLMInference/datasets'
RAW_DATASET     = f'{NFS_DATASET_DIR}/{DATASET_FILE}'
PROCESSED_DS    = f'/tmp/mlperf_{MODEL_NAME}_dataset.jsonl'

# ── Benchmark repo ───────────────────────────────────────────────────────────
INFERENCE_DIR   = '/tmp/corespan_inference'
# Map model names to the right benchmark directory in the repo
MODEL_BENCH_MAP = {
    'llama3.1-70B'   : 'llama2-70b',  # 70B uses same script
}
BENCH_DIR_NAME  = MODEL_BENCH_MAP.get(MODEL_NAME, 'llama3.1-8b')  # default
BENCH_DIR       = f'{INFERENCE_DIR}/language/{BENCH_DIR_NAME}'
LOG_DIR         = '/tmp/mlperf_llm_logs'

print(f'Model        : {MODEL_NAME}')
print(f'Model path   : {NFS_MODEL_PATH}')
print(f'Bench scripts: {BENCH_DIR}')
print(f'Dataset      : {RAW_DATASET}')
print(f'Scenario     : {SCENARIO}')
print(f'Samples      : {TOTAL_SAMPLES}')


## Cell 3 — Validate NFS Paths

In [ ]:
import os

assert os.path.isdir(NFS_MODEL_PATH), \
    f'Model not found: {NFS_MODEL_PATH}\nCheck MODEL_NAME in Cell 2'
assert os.path.isfile(RAW_DATASET), \
    f'Dataset not found: {RAW_DATASET}'

files = os.listdir(NFS_MODEL_PATH)
print(f'Model dir OK ({len(files)} files): {NFS_MODEL_PATH}')
print(f'Dataset OK  : {RAW_DATASET}  ({os.path.getsize(RAW_DATASET)//1024} KB)')


## Cell 4 — Clone corespan/inference (language only, sparse checkout)

In [ ]:
import os, subprocess

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out: print(out[-2000:])
    if r.returncode != 0: raise RuntimeError(f'Failed: {cmd}')

if os.path.isdir(BENCH_DIR):
    print(f'Already cloned: {BENCH_DIR}')
else:
    print('Sparse-cloning language benchmark ...')
    if not os.path.isdir(INFERENCE_DIR):
        run('git clone --depth 1 --filter=blob:none --sparse https://github.com/corespan/inference.git ' + INFERENCE_DIR)
    run(f'cd {INFERENCE_DIR} && git sparse-checkout add language/{BENCH_DIR_NAME}')
    print('Done.')


## Cell 5 — Install vLLM
The MLPerf LLM benchmark (`SUT_VLLM.py`) requires vLLM. Skipped if already installed.

In [ ]:
import importlib, subprocess, sys

if importlib.util.find_spec('vllm') is None:
    print('Installing vllm (this takes a few minutes)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'vllm==0.6.3', '--extra-index-url',
        'https://download.pytorch.org/whl/cu121'])
    print('vllm installed.')
else:
    import vllm
    print(f'vllm already installed: {vllm.__version__}')


## Cell 6 — Preprocess Dataset
The MLPerf `dataset.py` expects a JSONL with `input`, `tok_input`, `output` columns.
The raw OpenOrca JSONL on NFS needs tokenization first. Cached — skipped on re-runs.

In [ ]:
import os, json

if os.path.exists(PROCESSED_DS):
    count = sum(1 for _ in open(PROCESSED_DS))
    print(f'Processed dataset already exists: {PROCESSED_DS}  ({count} samples)')
else:
    from transformers import AutoTokenizer
    print(f'Loading tokenizer from {NFS_MODEL_PATH} ...')
    tok = AutoTokenizer.from_pretrained(NFS_MODEL_PATH, local_files_only=True)
    tok.padding_side = 'left'
    tok.pad_token    = tok.eos_token

    raw = []
    with open(RAW_DATASET) as f:
        for line in f:
            line = line.strip()
            if line: raw.append(json.loads(line))
    raw = raw[:TOTAL_SAMPLES]
    print(f'Loaded {len(raw)} raw samples. Tokenizing...')

    processed = []
    for s in raw:
        sys_p = s.get('system_prompt', '')
        q     = s.get('question', s.get('input', ''))
        ans   = s.get('response', s.get('output', ''))
        prompt = f'{sys_p}\n\nQuestion: {q}\nAnswer:' if sys_p else f'Question: {q}\nAnswer:'
        tok_input = tok.encode(prompt, add_special_tokens=True)
        processed.append({'input': prompt, 'tok_input': tok_input, 'output': ans})

    with open(PROCESSED_DS, 'w') as f:
        for row in processed:
            f.write(json.dumps(row) + '\n')
    print(f'Saved {len(processed)} processed samples to {PROCESSED_DS}')


## Cell 7 — Run MLPerf LLM Benchmark
Calls `main.py --vllm` from the cloned corespan/inference repo.

In [ ]:
import os, subprocess, time

os.makedirs(LOG_DIR, exist_ok=True)

cmd = (
    f'cd {BENCH_DIR} && '
    f'python main.py '
    f'--model-path {NFS_MODEL_PATH} '
    f'--dataset-path {PROCESSED_DS} '
    f'--scenario {SCENARIO} '
    f'--vllm '
    f'--dtype {DTYPE} '
    f'--tensor-parallel-size {TENSOR_PARALLEL} '
    f'--total-sample-count {TOTAL_SAMPLES} '
    f'--output-log-dir {LOG_DIR} '
    f'--num-workers 1'
)

print(f'Running:\n  {cmd}\n')
t0 = time.time()
r  = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0: print('STDERR:', r.stderr[-3000:])
print(f'Wall time: {time.time()-t0:.1f}s')


## Cell 8 — Results

In [ ]:
import os

summary = f'{LOG_DIR}/mlperf_log_summary.txt'
print('=== MLPerf Log Summary ===')
if os.path.exists(summary):
    print(open(summary).read())
else:
    print('Summary not found — check Cell 7 output for errors.')
